# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading and exploring the FAIR² Mental Health Survey dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

Croissant URL: [https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json](https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json)

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

# Print basic metadata
print(f"Dataset Name: {metadata.name}")
print(f"Description: {metadata.description}")
print(f"License: {metadata.license}")
print(f"Date Published: {metadata.datePublished}")
print(f"Version: {metadata.version}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

The Croissant schema organizes data into `recordSet` entities. Each `recordSet` has a unique `@id`, and fields, which may correspond to columns in the actual data files.

Let's enumerate all record sets, their fields and column `@id`s.

In [ ]:
# List record sets and fields using their @id
record_sets = dataset.record_sets

recordsets_ids = []
for rs in record_sets:
    print(f"RecordSet Name: {rs.name} | @id: {rs['@id']}")
    recordsets_ids.append(rs['@id'])
    if hasattr(rs, 'fields') and rs.fields:
        for field in rs.fields:
            print(f"    Field @id: {field['@id']} | Name: {field.name} | DataType: {field.dataType}")

## 3. Data Extraction
Load data from each record set into a DataFrame for analysis. Use the record set and field `@id`s above.

We'll load the main survey record set (named or designated as the primary table in the schema), but you can adapt for any record set using its `@id`.

In [ ]:
# Extract data from each record set into pandas DataFrames
dataframes = {}

# If there are multiple record sets, load each
for record_set_id in recordsets_ids:
    # Load records
    records = list(dataset.records(record_set=record_set_id))
    if len(records) > 0:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"\nRecordSet {record_set_id}: Columns: {df.columns.tolist()}")
        print(df.head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes operations such as removing outliers, transforming distributions, and grouping data by key attributes. All fields are referenced by their `@id` as required.

_Note_: Please update the `numeric_field_id` and `group_field_id` with actual field `@id` values based on your dataset overview.

In [ ]:
# Choose a record set and numeric field for analysis
record_set_id = recordsets_ids[0]  # Use the first record set for demonstration (update if needed)
df = dataframes[record_set_id]

# Example: Use GAD-7 total score, PHQ-9, or PSQ total as numeric fields
# Suppose their @id is 'http://mlcommons.org/croissant/#gad7_total', etc.

numeric_field_id = None
# Detect a likely numeric field (update this per actual dataset!):
for col in df.columns:
    if 'gad7' in col.lower() or 'phq9' in col.lower() or 'psq' in col.lower():
        numeric_field_id = col
        break

if numeric_field_id is None:
    # Choose a default field (update as needed)
    numeric_field_id = df.select_dtypes('number').columns[0]

print(f"Selected numeric field for EDA: {numeric_field_id}")

# Filter for records with numeric_field > threshold (e.g., GAD-7 score > 10)
threshold = 10
filtered_df = df[df[numeric_field_id] > threshold]
print(f"Filtered records with {numeric_field_id} > {threshold}:")
print(filtered_df.head())

# Normalize scores
filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
print(f"Normalized {numeric_field_id} for filtered records:")
print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Group by a demographic field: try 'age', 'gender', or similar (as @id from the schema)
group_field_id = None
for col in ['age', 'gender', 'marital_status', 'level_of_education', 'village']:
    for df_col in df.columns:
        if col in df_col.lower():
            group_field_id = df_col
            break
    if group_field_id: break

if group_field_id:
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
    print(f"Grouped data by {group_field_id} (mean {numeric_field_id}):")
    print(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

For example, plot the distribution of the selected numeric score and compare groups.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot the distribution of the selected numeric score
plt.figure(figsize=(8,6))
sns.histplot(df[numeric_field_id].dropna(), bins=20, kde=True)
plt.title(f"Distribution of {numeric_field_id}")
plt.xlabel(numeric_field_id)
plt.ylabel('Frequency')
plt.show()

if group_field_id:
    plt.figure(figsize=(8,6))
    sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
    plt.title(f"{numeric_field_id} by {group_field_id}")
    plt.xlabel(group_field_id)
    plt.ylabel(numeric_field_id)
    plt.xticks(rotation=45)
    plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- Loaded and previewed the dataset using the Croissant schema and `mlcroissant`.
- Explored record sets and fields via their `@id`.
- Performed basic filtering, normalization, and grouping based on survey scores and demographics.
- Visualized score distributions and demographic differences.

This notebook can be extended to perform deeper analyses such as correlation studies, predictive modeling, or integrating additional record sets.